# Patient Journey - Diagnosis (NBA 3.1)

Monthly CKD stage per patient from IQVIA LAAD Dx claims. The 3.0 query is kept untouched as a baseline (cell "Load dx data"); each step adds a new query and a row-by-row comparison with the previous one, the same approach as `01_patient_journey_labs`.

* **Plan and decisions:** `Patient_Journey_Validation/NBA_3_1_Dx_Journey_Plan.md`
* **Dx basket:** `Final Product Basket/Kerendia_Dx_Final.csv` (one row per code, stage per code). The original `Kerendia NBA 3.1 - DX Event List.csv` is unchanged.
* **Old notebook (3.0 baseline):** `(Clone) 02_patient_journey_dx` (unchanged)
* **Output:** `kerendia_nba_3_1_dx_journey_validation` is written only after review (Step 6). Until then section 2 still uses the 3.0 query.

## 1. Setup Snowflake connection and load data

In [0]:
%run "/Workspace/Users/wenjie.chen@bayer.com/kerendia_nba_3_0_develop/NBA 3.0 Ops - Pipeline/00_Helper_Notebooks/00_connection"

In [0]:
%run "/Workspace/Users/wenjie.chen@bayer.com/kerendia_nba_3_0_develop/NBA 3.0 Ops - Pipeline/00_Helper_Notebooks/00_config"

In [0]:
%run "../../00_cohort_config"

In [0]:
import pandas as pd
import pyspark.sql.functions as F
# Source: the 3.0 "Pat Journey Dx" query (00_Write_Input_Files), run directly from Snowflake.
# The pre-built query_output/pat_journey_dx table was removed in the 2026-10-26 refresh.
# Kept UNCHANGED as the baseline for all comparisons below (copied from (Clone) 02_patient_journey_dx).
print(f"Dx query window: {analysis_start_month} -> {current_month}")
dx_query = f"""
 SELECT DISTINCT
   NPI_NUMBER,
   PROVIDER_ID,
   PATIENT_ID,
   SVC_DT,
   CATEGORY as DX_STAGE_TEXT,
   CASE
      when CATEGORY = 'CHRONIC KIDNEY DISEASE, STAGE 1'  then 1
      when CATEGORY = 'CHRONIC KIDNEY DISEASE, STAGE 2 (MILD)'  then 2
      when CATEGORY = 'CHRONIC KIDNEY DISEASE, STAGE 3 (MODERATE)'  then 3
      when CATEGORY = 'CHRONIC KIDNEY DISEASE, STAGE 4 (SEVERE)'  then 4
      when CATEGORY like '%STAGE 5%'  then 5
      else 0
      end as DX_STAGE
 FROM
 (
  SELECT  DISTINCT
                   NPI_NUMBER,
                   PRIMARY_SPECIALTY_DESC,
                   PATIENT_ID,SERVICE_DATE AS SVC_DT,
                   CD.DIAGNOSIS_DESCRIPTION as Category,
                   COALESCE(PROVIDER_REFERRING_ID, PROVIDER_RENDERING_ID) as PROVIDER_ID
  FROM (Select
C.NPI_NUMBER,
A.PATIENT_ID,
A.SERVICE_DATE,
A.PROVIDER_RENDERING_ID,
A.PROVIDER_REFERRING_ID,
A.DIAGNOSIS_CODE,
B.PATIENT_BIRTH_YEAR,
C.PRIMARY_SPECIALTY_CODE,
C.PRIMARY_SPECIALTY_DESC
from  (select PATIENT_ID, CLAIM_ID, DIAGNOSIS_CODE, SERVICE_DATE, PROVIDER_RENDERING_ID, PROVIDER_REFERRING_ID  FROM
PHCDW.PHCDW_STG_CAR.STG_CAR_IQV_LAAD_FACT_DX) A
left outer join
(select distinct PATIENT_ID, PATIENT_BIRTH_YEAR, PATIENT_GENDER from PHCDW.PHCDW_STG_CAR.STG_CAR_IQV_LAAD_DIM_PTNT_DMGRPHC) B
on A.PATIENT_ID = B.PATIENT_ID
left outer join
PHCDW.PHCDW_STG_CAR.STG_CAR_IQV_LAAD_DIM_PRVDR C
on COALESCE(A.PROVIDER_REFERRING_ID,A.PROVIDER_RENDERING_ID) = C.PROVIDER_ID
) DX
  JOIN (
        SELECT DISTINCT DIAGNOSIS_CODE, DIAGNOSIS_DESCRIPTION
        FROM PHCDW.PHCDW_STG_CAR.STG_IQVIA_LAAD_DIM_DIAGNOSIS
        WHERE DIAGNOSIS_CODE IN (
          '249.4', '249.40', '249.41', '250.4', '250.40', '250.42', '285.21', '403', '403.0', '403.00', '403.1', '403.10', '403.9',
          '403.90', '404', '404.0', '404.00', '404.01', '404.1', '404.10', '404.11', '404.9', '404.90', '404.91', '585', '585.1',
          '585.2', '585.3', '585.4', '585.9', '588.81', 'D63.1', 'E08.2', 'E08.21','E08.22', 'E08.29', 'E09.2', 'E09.21', 'E09.22',
          'E09.29', 'E11.2', 'E11.21', 'E11.22', 'E11.29', 'E13.2', 'E13.21', 'E13.22', 'E13.29', 'I12', 'I12.9', 'I13', 'I13.0',
          'I13.1', 'I13.10', 'N18', 'N18.1', 'N18.2', 'N18.3', 'N18.4', 'N18.9', 'N25.81', '503.60' , '503.65', 'S20.65', '0TY0.0Z0',
          '0TY0.0Z1', '0TY0.0Z2', '0TY1.0Z0', '0TY1.0Z1', '0TY1.0Z2', 'Z94.0', 'V42.0', '556.9', '909.18', '909.19', '909.20', '909.21',
          '909.22', '909.23', '909.24', '909.25', '909.35', '909.37', '909.45', '909.47', '909.51', '909.52', '909.53', '909.54',
          '909.55', '909.56', '909.57', '909.58', '909.59', '909.60', '909.61', '909.62', '909.63', '909.64', '909.65', '909.66',
          '909.67', '909.68', '909.69', '909.70', '909.99', '995.12', '995.59', 'G02.57', 'G03.08', 'G03.09', 'G03.10', 'G03.11',
          'G03.12', 'G03.13', 'G03.14', 'G03.15', 'G03.16', 'G03.17', 'G03.18', 'G03.19', 'G03.20', 'G03.21', 'G03.22', 'G03.23',
          'G03.24', 'G03.25', 'G03.26', 'G03.27', 'G90.13', 'G90.14', 'S93.35', 'S93.39','3E1.M39Z', '5A1.D00Z', '5A1.D60Z',
          '5A1.D70Z', '5A1.D80Z', '5A1.D90Z', 'Z99.2', '399.5', '549.8', '800', '801', '802', '803', '804', '809', '820', '821',
          '829', '830', '831', '839', '840', '841', '849', '850', '851', '859', '880', '881', '889')
        ) CD
  ON DX.DIAGNOSIS_CODE=CD.DIAGNOSIS_CODE
   WHERE (EXTRACT(YEAR FROM SVC_DT)*100 + EXTRACT(MONTH FROM SVC_DT)) >= '{analysis_start_month}'
    AND (EXTRACT(YEAR FROM SVC_DT)*100 + EXTRACT(MONTH FROM SVC_DT)) <= '{current_month}' AND COALESCE(PROVIDER_REFERRING_ID,PROVIDER_RENDERING_ID) IS NOT NULL
  )
"""

df_dx = Get_Data_Snowflakes(dx_query)

# HCP universe (same as 3.0 00_Write_Input_Files)
_hcp_universe = spark.read.format("delta").load(hcp_universe_path).select("PROVIDER_ID").distinct()
df_dx = df_dx.join(_hcp_universe, how="inner", on="PROVIDER_ID")

# Filter to data >= DATA_START_DATE (from config)
df_dx = df_dx.filter(F.col("SVC_DT") >= DATA_START_DATE)

# Validation filter: 10 test HCPs (from config)
df_dx, test_patients = apply_hcp_filter(df_dx)

pd.DataFrame(df_dx.columns).display()

In [0]:
# The 3.0 monthly processing (old notebook cells 12, 13, 16), wrapped in a function so the same logic can be
# applied to every query version in the comparisons below. Logic is unchanged:
#   global grid (all patients x all months between the first and last SVC_DT), max stage per month,
#   stage 0 -> null, forward fill, null -> 0, cohort label.
from datetime import datetime, timedelta
from pyspark.sql.types import StructType, StructField, StringType
from pyspark.sql import Window
import sys

def generate_month_range(start, end):
    current = datetime.strptime(start, "%Y-%m-%d")
    end = datetime.strptime(end, "%Y-%m-%d")
    months = []
    while current <= end:
        months.append(current.strftime("%Y-%m-01"))
        if current.month == 12:
            current = current.replace(year=current.year + 1, month=1, day=1)
        else:
            current = current.replace(month=current.month + 1, day=1)
    return months

def dx_monthly_30(df):
    """df needs PATIENT_ID, SVC_DT, DX_STAGE. Returns PATIENT_ID, DATE, DX_STAGE (array), DX_STAGE_UPDATED, cohort_label."""
    min_dt, max_dt = df.agg(F.min("SVC_DT"), F.max("SVC_DT")).first()
    if OFFSET_DAY == 0:
        min_month_str = min_dt.strftime("%Y-%m-01")
        max_month_str = max_dt.strftime("%Y-%m-01")
    else:
        min_month_str = (min_dt - timedelta(days=OFFSET_DAY - 1)).strftime("%Y-%m-01")
        max_month_str = (max_dt - timedelta(days=OFFSET_DAY - 1)).strftime("%Y-%m-01")

    all_months = generate_month_range(min_month_str, max_month_str)
    months_df = spark.createDataFrame([(m,) for m in all_months],
                                      schema=StructType([StructField("COHORT_MONTH", StringType(), True)]))
    patient_months_df = df.select("PATIENT_ID").distinct().crossJoin(months_df)

    monthly = (df.withColumn("COHORT_MONTH", F.date_format(assign_cohort_month(F.col("SVC_DT")), "yyyy-MM-01"))
                 .groupBy("PATIENT_ID", "COHORT_MONTH")
                 .agg(F.collect_set("DX_STAGE").alias("DX_STAGE")))
    monthly = patient_months_df.join(monthly, on=["PATIENT_ID", "COHORT_MONTH"], how="left")
    monthly = (monthly.withColumn("DX_STAGE_MAX", F.array_max(F.col("DX_STAGE")))
                      .withColumn("DX_STAGE_MAX", F.when(F.col("DX_STAGE_MAX") == 0, None).otherwise(F.col("DX_STAGE_MAX"))))
    w_forward = Window.partitionBy("PATIENT_ID").orderBy("COHORT_MONTH").rowsBetween(-sys.maxsize, 0)
    filled = monthly.withColumn("DX_STAGE_UPDATED", F.last("DX_STAGE_MAX", ignorenulls=True).over(w_forward))
    combine = (monthly.join(filled.select("PATIENT_ID", "COHORT_MONTH", "DX_STAGE_UPDATED"),
                            on=["PATIENT_ID", "COHORT_MONTH"], how="left")
                      .na.fill(value=0, subset=["DX_STAGE_UPDATED"]))
    combine = combine.join(
        df_cohort_ref.select(F.date_format("cohort_ref_month", "yyyy-MM-01").alias("_ref_month_str"), "cohort_label"),
        combine.COHORT_MONTH == F.col("_ref_month_str"), "left").drop("_ref_month_str")
    return (combine.withColumnRenamed("COHORT_MONTH", "DATE").drop("DX_STAGE_MAX")
                   .withColumn("DATE", F.to_date(F.col("DATE"), "yyyy-MM-dd")))

print("dx_monthly_30 defined")

In [0]:
# Step 0 check: the 3.0 query + 3.0 processing in this notebook must give the same output as the table
# last written by the old notebook ((Clone) 02_patient_journey_dx, cell "Write dx journey validation table").
# If that table was written with different config dates, differences show up here and are explained, not hidden.
DX_VALIDATION_PATH = ("/Volumes/ph-com-ai-us-prod-catalog-2553213839599380/kerendia_nba_3/kerendia_nba_3_volume/"
                      "hcp_level_model/ops_pipeline/query_output/kerendia_nba_3_1_dx_journey_validation")

df_dx_monthly_30 = dx_monthly_30(df_dx)
try:
    _old = spark.read.format("delta").load(DX_VALIDATION_PATH)
except Exception as e:
    _old = None
    print(f"Old validation table not readable, Step 0 check skipped: {str(e)[:200]}")

if _old is not None:
    _k = ["PATIENT_ID", "DATE"]
    _new = df_dx_monthly_30.select(*_k, F.col("DX_STAGE_UPDATED").alias("NEW"), F.col("cohort_label").alias("NEW_LBL"))
    _o = _old.select(F.col("PATIENT_ID").cast("string").alias("PATIENT_ID"), F.col("DATE").cast("date").alias("DATE"),
                     F.col("DX_STAGE_UPDATED").alias("OLD"), F.col("cohort_label").alias("OLD_LBL"))
    _j = _new.withColumn("PATIENT_ID", F.col("PATIENT_ID").cast("string")).join(_o, _k, "full")
    _s = _j.agg(
        F.count("*").alias("rows_union"),
        F.sum(F.when(F.col("OLD").isNull() & F.col("OLD_LBL").isNull() & F.col("NEW").isNotNull(), 1).otherwise(0)).alias("only_new"),
        F.sum(F.when(F.col("NEW").isNull() & F.col("OLD").isNotNull(), 1).otherwise(0)).alias("only_old"),
        F.sum(F.when(F.col("NEW") == F.col("OLD"), 1).otherwise(0)).alias("same_stage"),
        F.sum(F.when(F.col("NEW") != F.col("OLD"), 1).otherwise(0)).alias("different_stage"),
        F.sum(F.when(~F.col("NEW_LBL").eqNullSafe(F.col("OLD_LBL")) & F.col("NEW").isNotNull() & F.col("OLD").isNotNull(), 1).otherwise(0)).alias("different_label"),
    ).toPandas()
    print(f"This notebook: {df_dx_monthly_30.count():,} patient-months, {df_dx_monthly_30.select('PATIENT_ID').distinct().count():,} patients")
    print(f"Old table:     {_old.count():,} patient-months, {_old.select('PATIENT_ID').distinct().count():,} patients")
    display(_s)

### Step 1: manual (hardcoded) parts of the 3.0 Dx query and processing

Review of the cell "Load dx data" and the 3.0 monthly processing. Full detail in the plan file, section 2.

| # | Hardcoded today | Effect | Fixed in |
| --- | --- | --- | --- |
| Q1 | 163-code list typed in the SQL | Adding a code means editing the SQL | Step 2 (basket) |
| Q2 | 131 of the 163 codes cannot match Dx data: 32 ICD-9 codes and 99 dialysis / transplant procedure codes written with a dot (`909.xx`, `G03.xx`, `5A1.D00Z`, `800`-`889`) | Hide what the list covers | Step 2 (basket); dialysis / transplant intent belongs in Px |
| Q3 | Stage from the description text (`CATEGORY = 'CHRONIC KIDNEY DISEASE, STAGE 1'` ...) | `N18.30/31/32` and all stage-5 codes are not in the list: almost no stage 3, no stage 5 | Step 2 (stage per code in the basket) |
| Q4 | Header codes (`N18`, `I12`, `E11.2` ...) and `E09.21`, `E09.29`, `N25.81`, `Z94.0` | Not in the 3.1 basket | Step 2 (measured) |
| Q6 | Date filter `YEAR*100 + MONTH` on every row, plus `DATA_START_DATE` in Spark | Snowflake cannot skip data by date; two date settings | Step 3 (config) |
| Q7 | Patient demographics join never used | Extra cost | Step 3 |
| Q8, Q9 | Table / column names and the provider rule `COALESCE(REFERRING, RENDERING)` typed in the SQL | A rename breaks the query | Step 3 (`DX_CONFIG`) |
| Q10, Q11 | `DIAGNOSIS_CODE` not in the output; no dropped-rows report | No QC by code; silent losses | Steps 2 and 4 |
| C1, C3 | Global grid: every patient x every month from the first to the last service date | Very large at all-US scale; rows before the first diagnosis | Step 5 (per-patient range to `date_end`) |
| C2 | 0 means both "CKD, stage unknown" and "before any diagnosis"; in PJ06 the 0 hides the lab stage | Wrong combined stage | Step 5 (null + `DX_CKD_FLAG`) |
| C5 | `DX_STAGE` output is an array | Hard to use | Step 5 |

## Step 2: Basket changes (Q1-Q4, Q10)

The code list and the stage now come from `Final Product Basket/Kerendia_Dx_Final.csv`: only codes with `ACTIVE_STAGING = Y` are used (decision D1), and the stage is read per code from `CKD_STAGE` instead of the description text. Everything else (joins, date filter, provider rule, post-processing) is kept exactly as in 3.0, so the comparison shows only the effect of the basket.

| Cell | What it does |
| --- | --- |
| 2a | Reads the basket, expands clubbed rows, checks it, and checks it has the same codes as the original Dx Event List |
| 2b | **The Dx query** (built from the basket) |
| 2c | Runs the query with the same post-processing as the 3.0 cell |
| 2d | Compares the 3.0 query with the basket query (all US and validation HCPs) |
| 2e | Findings |

In [0]:
# 2a. Read the Dx basket and check it is valid before building the query.
#
# Basket file: one row per code (MATCH_TYPE = CODE) or per clubbed group (MATCH_TYPE = LIST, member codes separated by |)
#   CONDITION       CKD / T2D / T1D / OTHER_DM / HF / DIALYSIS / TRANSPLANT / ...  (what the code is about)
#   ACTIVE_STAGING  Y = used by this CKD journey (decision D1)
#   CKD_STAGE       1-5, 0 = CKD stage unknown, empty = not staged        CKD_SUBSTAGE  3a / 3b / ESRD
#   DM_TYPE         T2D / T1D / OTHER (Phase B)    HF_TYPE  HF type proxy (Phase C)    FLAG  HF / DIALYSIS / TRANSPLANT / ...
import pandas as pd

_BASKET_DIR = ("/Workspace/Users/wenjie.chen@bayer.com/kerendia_nba_3_1_develop_sm/NBA 3.1 - Dev/"
               "03_Feature_Engineering/03_HCP_level_features/Final Product Basket")
DX_BASKET_PATH = f"{_BASKET_DIR}/Kerendia_Dx_Final.csv"
DX_EVENT_LIST_PATH = f"{_BASKET_DIR}/Kerendia NBA 3.1 - DX Event List.csv"   # original list, only used for the check below

dx_basket_raw = pd.read_csv(DX_BASKET_PATH, dtype=str).fillna("")
# one row per real code: LIST rows are expanded into their member codes
dx_basket = (dx_basket_raw.assign(DIAGNOSIS_CODE=dx_basket_raw["MATCH_VALUE"].str.split("|"))
             .explode("DIAGNOSIS_CODE").reset_index(drop=True))
dx_basket["DIAGNOSIS_CODE"] = dx_basket["DIAGNOSIS_CODE"].str.strip()

# Checks: the cell stops with a clear message if the file is not valid
_raw = dx_basket_raw
assert set(_raw["MATCH_TYPE"]) <= {"CODE", "LIST"}, "MATCH_TYPE must be CODE or LIST"
assert (~_raw.loc[_raw.MATCH_TYPE == "CODE", "MATCH_VALUE"].str.contains(r"\|")).all(), "A CODE row has several codes"
assert (_raw.loc[_raw.MATCH_TYPE == "LIST", "CLUB_ID"] != "").all(), "A LIST row has no CLUB_ID"
assert not dx_basket["DIAGNOSIS_CODE"].duplicated().any(), \
    f"Codes listed twice: {dx_basket.loc[dx_basket.DIAGNOSIS_CODE.duplicated(), 'DIAGNOSIS_CODE'].tolist()}"
assert set(dx_basket["ACTIVE_STAGING"]) <= {"Y", "N"}, "ACTIVE_STAGING must be Y or N"
_staged = dx_basket[dx_basket["ACTIVE_STAGING"] == "Y"]
assert _staged["CKD_STAGE"].isin(["0", "1", "2", "3", "4", "5"]).all(), "Staged codes need CKD_STAGE 0-5"
assert (dx_basket.loc[dx_basket.ACTIVE_STAGING == "N", ["CKD_STAGE", "CKD_SUBSTAGE"]] == "").all().all(), \
    "Codes not used for staging must have empty CKD_STAGE / CKD_SUBSTAGE"
assert set(dx_basket["CKD_SUBSTAGE"]) <= {"", "3a", "3b", "ESRD"}, "CKD_SUBSTAGE must be 3a, 3b or ESRD"
assert set(dx_basket["DM_TYPE"]) <= {"", "T1D", "T2D", "OTHER"}, "DM_TYPE must be T1D, T2D or OTHER"
assert set(dx_basket["HF_TYPE"]) <= {"", "HFPEF_PROXY", "HFREF_PROXY", "COMBINED", "UNKNOWN"}, "Unknown HF_TYPE"
assert set(dx_basket["FLAG"]) <= {"", "HF", "DIALYSIS", "TRANSPLANT", "PROTEINURIA", "KIDNEY_CV"}, "Unknown FLAG"
assert ((dx_basket["HF_TYPE"] != "") == (dx_basket["FLAG"] == "HF")).all(), "HF_TYPE must be set exactly on HF-flag codes"

# The basket must have exactly the codes of the original Dx Event List (nothing lost or added in the rebuild),
# with the same patient counts and flags
_orig = pd.read_csv(DX_EVENT_LIST_PATH, dtype=str).fillna("")
_club = _orig["is_clubbed"].str.upper() == "YES"
_orig_codes = set(_orig.loc[~_club, "DIAGNOSIS_CODE"]) | {m.strip() for x in _orig.loc[_club, "club_members"] for m in x.split(",") if m.strip()}
_missing, _extra = _orig_codes - set(dx_basket.DIAGNOSIS_CODE), set(dx_basket.DIAGNOSIS_CODE) - _orig_codes
assert not _missing and not _extra, f"Codes differ from the original list. Missing: {sorted(_missing)}; extra: {sorted(_extra)}"
_key = _raw["MATCH_VALUE"].where(_raw.MATCH_TYPE == "CODE", _raw["CLUB_ID"] + "_CLUBBED")
_cmp = (_raw.assign(DIAGNOSIS_CODE=_key)
        .merge(_orig, on="DIAGNOSIS_CODE", how="outer", indicator=True))
_bad = _cmp[(_cmp._merge != "both")
            | (_cmp.PATIENTS_TOTAL != _cmp.patients_total)
            | ((_cmp.IS_DEFINING == "Y") != (_cmp.is_defining.str.upper() == "YES"))
            | ((_cmp.IN_752_FEATURES == "Y") != (_cmp.in_752_features.str.upper() == "YES"))]
assert _bad.empty, f"Rows differ from the original list: {_bad.DIAGNOSIS_CODE.tolist()}"

print(f"Basket OK: {len(_raw)} rows ({(_raw.MATCH_TYPE == 'LIST').sum()} clubbed) = {len(dx_basket)} codes, "
      f"same codes as the original Dx Event List ({len(_orig)} rows)")
print(f"Used for staging (ACTIVE_STAGING = Y): {len(_staged)} codes")
display(_staged.groupby("CKD_STAGE").agg(codes=("DIAGNOSIS_CODE", lambda s: ", ".join(s)), n=("DIAGNOSIS_CODE", "size")).reset_index())
display(dx_basket.groupby(["CONDITION", "ACTIVE_STAGING"]).size().rename("codes").reset_index())

In [0]:
# 2b. The Dx query, built from the basket (2a). Same joins, date filter and provider rule as the 3.0 query;
# the code list and the stage per code come from the basket. DIAGNOSIS_CODE is now kept in the output.
# Dates, table names and the unused demographics join are still as in 3.0 (Step 3).

def build_dx_query_basket(basket):
    staged = basket[basket["ACTIVE_STAGING"] == "Y"]
    values = ",\n      ".join(f"('{r.DIAGNOSIS_CODE}', {int(r.CKD_STAGE)})" for r in staged.itertuples())
    return f"""
SELECT DISTINCT NPI_NUMBER, PROVIDER_ID, PATIENT_ID, SVC_DT, DIAGNOSIS_CODE, DX_STAGE
FROM (
  SELECT C.NPI_NUMBER,
         A.PATIENT_ID,
         A.SERVICE_DATE AS SVC_DT,
         A.DIAGNOSIS_CODE,
         BSK.CKD_STAGE AS DX_STAGE,
         COALESCE(A.PROVIDER_REFERRING_ID, A.PROVIDER_RENDERING_ID) AS PROVIDER_ID
  FROM (SELECT PATIENT_ID, CLAIM_ID, DIAGNOSIS_CODE, SERVICE_DATE, PROVIDER_RENDERING_ID, PROVIDER_REFERRING_ID
        FROM PHCDW.PHCDW_STG_CAR.STG_CAR_IQV_LAAD_FACT_DX) A
  JOIN (SELECT COLUMN1 AS DIAGNOSIS_CODE, COLUMN2 AS CKD_STAGE FROM VALUES
      {values}
  ) BSK ON A.DIAGNOSIS_CODE = BSK.DIAGNOSIS_CODE
  LEFT OUTER JOIN (SELECT DISTINCT PATIENT_ID, PATIENT_BIRTH_YEAR, PATIENT_GENDER
                   FROM PHCDW.PHCDW_STG_CAR.STG_CAR_IQV_LAAD_DIM_PTNT_DMGRPHC) B ON A.PATIENT_ID = B.PATIENT_ID
  LEFT OUTER JOIN PHCDW.PHCDW_STG_CAR.STG_CAR_IQV_LAAD_DIM_PRVDR C
    ON COALESCE(A.PROVIDER_REFERRING_ID, A.PROVIDER_RENDERING_ID) = C.PROVIDER_ID
  WHERE (EXTRACT(YEAR FROM A.SERVICE_DATE)*100 + EXTRACT(MONTH FROM A.SERVICE_DATE)) >= '{analysis_start_month}'
    AND (EXTRACT(YEAR FROM A.SERVICE_DATE)*100 + EXTRACT(MONTH FROM A.SERVICE_DATE)) <= '{current_month}'
    AND COALESCE(A.PROVIDER_REFERRING_ID, A.PROVIDER_RENDERING_ID) IS NOT NULL
)
"""

dx_query_basket = build_dx_query_basket(dx_basket)
print(dx_query_basket)

In [0]:
# 2c. Run the basket query with the same post-processing as the 3.0 cell
df_dx_basket_all = Get_Data_Snowflakes(dx_query_basket)
df_dx_basket = df_dx_basket_all.join(_hcp_universe, how="inner", on="PROVIDER_ID")
df_dx_basket = df_dx_basket.filter(F.col("SVC_DT") >= DATA_START_DATE)
df_dx_basket, _ = apply_hcp_filter(df_dx_basket)
print(f"Validation HCPs: {df_dx_basket.count():,} rows, {df_dx_basket.select('PATIENT_ID').distinct().count():,} patients "
      f"(3.0 query: {df_dx.count():,} rows, {df_dx.select('PATIENT_ID').distinct().count():,} patients)")

In [0]:
# 2d. Compare the 3.0 query with the basket query.
# Part A: all US, 2025-01 to 2026-09, in Snowflake. Grain = patient x day, highest stage of the day
#         (the 3.0 processing keeps the highest stage per month and ignores the provider).
# Part B: validation HCPs, month by month, both queries through the same 3.0 processing (dx_monthly_30).
import re
OLD_CODES = re.findall(r"'([^']+)'", re.search(r"DIAGNOSIS_CODE IN \((.*?)\)", dx_query, re.S).group(1))
CMP_START, CMP_END = "2025-01-01", "2026-09-30"

def _day_stage(q):
    return (f"SELECT PATIENT_ID, SVC_DT, MAX(DX_STAGE) AS STAGE FROM ({q}) "
            f"WHERE SVC_DT BETWEEN '{CMP_START}' AND '{CMP_END}' GROUP BY 1, 2")

# Part A1: stage of each patient-day, 3.0 vs basket
_a1 = Get_Data_Snowflakes(f"""
WITH O AS ({_day_stage(dx_query)}), N AS ({_day_stage(dx_query_basket)})
SELECT O.STAGE AS STAGE_30, N.STAGE AS STAGE_NEW, COUNT(*) AS PATIENT_DAYS
FROM O FULL OUTER JOIN N ON O.PATIENT_ID = N.PATIENT_ID AND O.SVC_DT = N.SVC_DT
GROUP BY 1, 2""").toPandas()
_a1["STAGE_30"] = _a1["STAGE_30"].astype("Int64")
_a1["STAGE_NEW"] = _a1["STAGE_NEW"].astype("Int64")
_a1["RESULT"] = ["lost" if pd.isna(n) else "gained" if pd.isna(o) else "same" if o == n else "different"
                 for o, n in zip(_a1.STAGE_30, _a1.STAGE_NEW)]
def _show_a1(a1):
    print("Part A1 - all US, 2025-01 to 2026-09, patient-days")
    display(a1.groupby("RESULT")["PATIENT_DAYS"].sum().reset_index())
    _lbl = lambda s: s.astype("string").fillna("not found")
    display(a1.assign(STAGE_30=_lbl(a1.STAGE_30), STAGE_NEW=_lbl(a1.STAGE_NEW), PATIENT_DAYS=a1.PATIENT_DAYS.astype("int64"))
              .pivot_table(index="STAGE_30", columns="STAGE_NEW", values="PATIENT_DAYS", aggfunc="sum", fill_value=0)
              .rename_axis(index="3.0 stage (rows) / basket stage (columns)").reset_index())
_show_a1(_a1)

# Part A2: basket rows by code (same window), marking codes that were not in the 3.0 list
_a2 = Get_Data_Snowflakes(f"""
SELECT DIAGNOSIS_CODE, DX_STAGE, COUNT(*) AS ROWS_, COUNT(DISTINCT PATIENT_ID) AS PATIENTS
FROM ({dx_query_basket}) WHERE SVC_DT BETWEEN '{CMP_START}' AND '{CMP_END}' GROUP BY 1, 2""").toPandas()
_a2["IN_30_LIST"] = _a2["DIAGNOSIS_CODE"].isin(OLD_CODES)
print("Part A2 - basket query by code, all US, 2025-01 to 2026-09")
display(_a2.sort_values(["IN_30_LIST", "PATIENTS"], ascending=[True, False]))

# Part A3: 3.0 patient-days not found by the basket query, by 3.0 description
_a3 = Get_Data_Snowflakes(f"""
WITH O AS (SELECT DISTINCT PATIENT_ID, SVC_DT, DX_STAGE_TEXT FROM ({dx_query}) WHERE SVC_DT BETWEEN '{CMP_START}' AND '{CMP_END}'),
     N AS (SELECT DISTINCT PATIENT_ID, SVC_DT FROM ({dx_query_basket}) WHERE SVC_DT BETWEEN '{CMP_START}' AND '{CMP_END}')
SELECT O.DX_STAGE_TEXT, COUNT(*) AS PATIENT_DAYS, COUNT(DISTINCT O.PATIENT_ID) AS PATIENTS
FROM O LEFT JOIN N ON O.PATIENT_ID = N.PATIENT_ID AND O.SVC_DT = N.SVC_DT
WHERE N.PATIENT_ID IS NULL GROUP BY 1 ORDER BY 2 DESC""").toPandas()
print("Part A3 - 3.0 patient-days lost in the basket query, by 3.0 description")
display(_a3)

# Part B: validation HCPs, month by month
_m30 = dx_monthly_30(df_dx).select("PATIENT_ID", "DATE", F.col("DX_STAGE_UPDATED").cast("int").alias("S30"))
_mnew = dx_monthly_30(df_dx_basket).select("PATIENT_ID", "DATE", F.col("DX_STAGE_UPDATED").cast("int").alias("SNEW"))
_mb = _m30.join(_mnew, ["PATIENT_ID", "DATE"], "full")
print("Part B - validation HCPs, patient-months")
print(f"  3.0: {_m30.count():,} patient-months, {_m30.select('PATIENT_ID').distinct().count():,} patients | "
      f"basket: {_mnew.count():,} patient-months, {_mnew.select('PATIENT_ID').distinct().count():,} patients")
display(_mb.groupBy("S30").pivot("SNEW").count().orderBy("S30"))

def _patients_by_stage(m, col):
    # patients who reach each stage in at least one month
    return m.filter(F.col(col) > 0).groupBy(col).agg(F.countDistinct("PATIENT_ID").alias("patients")).withColumnRenamed(col, "STAGE")
display(_patients_by_stage(_m30, "S30").withColumnRenamed("patients", "patients_30")
        .join(_patients_by_stage(_mnew, "SNEW").withColumnRenamed("patients", "patients_basket"), "STAGE", "full")
        .orderBy("STAGE"))

### 2e. Findings: basket query vs 3.0 query (run 2026-10-29)

**Basket (2a).** 240 rows (5 clubbed) = 368 codes, exactly the codes of the original Dx Event List, with the same patient counts and flags. 31 codes are used for staging: stage 0 (19 codes), 1, 2, 4 (1 each), 3 (4 codes), 5 (5 codes).

**Part A, all US, 2025-01 to 2026-09 (patient-days, highest stage of the day)**

| Result | Patient-days | What it is |
| --- | --- | --- |
| Same stage | 70.86M | |
| Different stage | 79.42M | All **upward**: 0 → 5 (52.19M), 0 → 3 (26.77M), 4 → 5 (344K), 2 → 3 (92K), 2 → 5, 1 → 3, 1 → 5, 3 → 5. No stage goes down, as expected: the basket keeps the 3.0 CKD codes and adds the stage-3 / stage-5 codes |
| Gained | 37.11M | Patient-days with no 3.0 code: stage 3 (22.71M), stage 5 (13.99M), stage 0 (404K, the T1D kidney codes E10.21/22/29, decision D2) |
| Lost | 2,612 | 3.0 codes not in the 3.1 basket: E09.21 / E09.29 (drug-induced diabetes with kidney disease, 1,755 days, up to 734 patients), header codes N18 / E11.2 / I12 / I13 / E08.2 (~850 days), and a few ICD-9 rows. All were stage 0 in 3.0 |

0 → 5 is the largest change because ESRD patients (N18.6, 69.4M rows, 1.12M patients) had only stage-0 codes on the same day in 3.0 (for example Z99.2 or D63.1, which are billed together with N18.6) and are now stage 5 (decision D3).

**New codes found (Part A2, patients 2025-01 to 2026-09):** N18.31 (4.87M), N18.32 (3.09M), N18.30 (2.82M), N18.6 (1.12M), I12.0 (665K), I13.2 (393K), N18.5 (365K), E10.22 (120K), I13.11 (63K), E10.21 (43K), E10.29 (33K). The old header code N18.3 has only 3.4K patients.

**Part B, validation HCPs (10 doctors)**

| | 3.0 | Basket |
| --- | --- | --- |
| Patients | 817 | 1,163 (+346 who only had codes missing from the 3.0 list) |
| Patient-months | 37,582 | 53,498 |
| Patients reaching stage 3 | 0 | 690 |
| Patients reaching stage 5 | 0 | 326 |
| Patients reaching stage 4 | 257 | 288 |

The stage-3 and stage-5 counts match the expected impact in the proposals doc (0 → 690 and 0 → 326). No 3.0 patient-month is lost.

**Observations for review**
* **E09.21 and E09.29** (drug-induced diabetes with nephropathy / other kidney complication) were in the 3.0 list but are not in the 3.1 Dx Event List. They are stage 0 (no effect on the stage), but up to 734 patients (2025-01 to 2026-09, all US) lose these patient-days. Candidate to add to the basket as stage 0, like E08.21/29 and E13.21/29.
* **Latest code wins, not highest.** 906 validation patient-months go from 4 (3.0) to 3: the 3.0 forward fill keeps the most recent stage, and a later N18.3x code now replaces an older stage 4. This is the 3.0 rule, unchanged here; it can be reviewed in Step 5.

## Step 3: Config changes (Q6-Q9)

Run settings move into one block, `DX_CONFIG`: start and end dates (applied on `SERVICE_DATE` inside the SQL, decision 5), source and provider table names with column mappings, and the provider rule `COALESCE(REFERRING, RENDERING)` (decision N3). The unused patient demographics join is removed. The basket logic is the same as Step 2.

| Cell | What it does |
| --- | --- |
| 3a | `DX_CONFIG` and checks that tables and columns exist |
| 3b | **The Dx query** (basket + config) |
| 3c | Runs it (no Spark date filter needed any more) |
| 3d | Compares with Step 2 |
| 3e | Findings |

In [0]:
# 3a. Run settings for the Dx query. Change values here, not in the SQL.
DX_CONFIG = {
    "start_date": str(DATA_START_DATE)[:10],     # from 00_cohort_config (3.0 used analysis_start_month in SQL + DATA_START_DATE in Spark)
    "end_date": str(date_end)[:10],              # from 00_config (3.0 used current_month, i.e. the whole month)
    "source": {
        "table": "PHCDW.PHCDW_STG_CAR.STG_CAR_IQV_LAAD_FACT_DX",
        "columns": {"patient_id": "PATIENT_ID", "service_date": "SERVICE_DATE", "diagnosis_code": "DIAGNOSIS_CODE",
                    "referring": "PROVIDER_REFERRING_ID", "rendering": "PROVIDER_RENDERING_ID"},
    },
    "provider": {
        "table": "PHCDW.PHCDW_STG_CAR.STG_CAR_IQV_LAAD_DIM_PRVDR",
        "columns": {"provider_id": "PROVIDER_ID", "npi": "NPI_NUMBER"},
    },
    # provider of a claim = first non-empty of these source columns (decision N3: referring first, then rendering)
    "provider_rule": ["referring", "rendering"],
}

# Checks: tables exist and every mapped column is there
for _blk in ("source", "provider"):
    _t = DX_CONFIG[_blk]["table"]
    _cols = set(c.upper() for c in Get_Data_Snowflakes(f"SELECT * FROM {_t} LIMIT 0").columns)
    _miss = [c for c in DX_CONFIG[_blk]["columns"].values() if c.upper() not in _cols]
    assert not _miss, f"{_t}: columns not found {_miss}"
assert all(k in DX_CONFIG["source"]["columns"] for k in DX_CONFIG["provider_rule"]), "provider_rule uses an unknown column key"
assert DX_CONFIG["start_date"] <= DX_CONFIG["end_date"], "start_date is after end_date"
print(f"DX_CONFIG OK: {DX_CONFIG['start_date']} -> {DX_CONFIG['end_date']}, tables and columns found")
print(f"(3.0 window was months {analysis_start_month} -> {current_month}, then SVC_DT >= {DATA_START_DATE} in Spark)")

In [0]:
# 3b. The Dx query: basket (Step 2) + settings from DX_CONFIG. Same output columns as Step 2.

def _provider_sql(cfg, alias="A"):
    cols = cfg["source"]["columns"]
    return "COALESCE(" + ", ".join(f"{alias}.{cols[k]}" for k in cfg["provider_rule"]) + ")"

def _basket_values(basket, extra_cols=()):
    staged = basket[basket["ACTIVE_STAGING"] == "Y"]
    rows = []
    for r in staged.itertuples():
        vals = [f"'{r.DIAGNOSIS_CODE}'", str(int(r.CKD_STAGE))] + [f"'{getattr(r, c)}'" for c in extra_cols]
        rows.append("(" + ", ".join(vals) + ")")
    names = ["COLUMN1 AS DIAGNOSIS_CODE", "COLUMN2 AS CKD_STAGE"] + [f"COLUMN{i + 3} AS {c}" for i, c in enumerate(extra_cols)]
    return "SELECT " + ", ".join(names) + " FROM VALUES\n      " + ",\n      ".join(rows)

def build_dx_query(basket, cfg):
    s, p = cfg["source"], cfg["provider"]
    sc, pc = s["columns"], p["columns"]
    prov = _provider_sql(cfg)
    return f"""
SELECT DISTINCT C.{pc['npi']} AS NPI_NUMBER,
       {prov} AS PROVIDER_ID,
       A.{sc['patient_id']} AS PATIENT_ID,
       A.{sc['service_date']} AS SVC_DT,
       A.{sc['diagnosis_code']} AS DIAGNOSIS_CODE,
       BSK.CKD_STAGE AS DX_STAGE
FROM {s['table']} A
JOIN ({_basket_values(basket)}
) BSK ON A.{sc['diagnosis_code']} = BSK.DIAGNOSIS_CODE
LEFT JOIN {p['table']} C ON {prov} = C.{pc['provider_id']}
WHERE A.{sc['service_date']} BETWEEN '{cfg['start_date']}' AND '{cfg['end_date']}'
  AND {prov} IS NOT NULL
"""

dx_query_config = build_dx_query(dx_basket, DX_CONFIG)
print(dx_query_config)

In [0]:
# 3c. Run the config query. Dates are applied in the SQL, so only the HCP universe and the validation filter remain in Spark.
df_dx_config_all = Get_Data_Snowflakes(dx_query_config)
df_dx_config = df_dx_config_all.join(_hcp_universe, how="inner", on="PROVIDER_ID")
df_dx_config, _ = apply_hcp_filter(df_dx_config)
print(f"Validation HCPs: {df_dx_config.count():,} rows, {df_dx_config.select('PATIENT_ID').distinct().count():,} patients "
      f"(Step 2: {df_dx_basket.count():,} rows, {df_dx_basket.select('PATIENT_ID').distinct().count():,} patients)")

In [0]:
# 3d. Compare Step 3 (config) with Step 2 (basket).
# Part A: all US, 2025-01 to 2026-09, row by row (provider, patient, day, code, stage), in Snowflake
# Part B: validation HCPs, row by row, in Spark
# Part C: rows pulled from Snowflake by each query (full window of each query)
_KEYS = ["NPI_NUMBER", "PROVIDER_ID", "PATIENT_ID", "SVC_DT", "DIAGNOSIS_CODE", "DX_STAGE"]
_on = " AND ".join(f"EQUAL_NULL(O.{k}, N.{k})" for k in _KEYS)
_w = f"WHERE SVC_DT BETWEEN '{CMP_START}' AND '{CMP_END}'"
_3a = Get_Data_Snowflakes(f"""
WITH O AS (SELECT * FROM ({dx_query_basket}) {_w}), N AS (SELECT * FROM ({dx_query_config}) {_w})
SELECT CASE WHEN O.PATIENT_ID IS NULL THEN 'only Step 3' WHEN N.PATIENT_ID IS NULL THEN 'only Step 2' ELSE 'same' END AS RESULT,
       CASE WHEN COALESCE(O.SVC_DT, N.SVC_DT) > '{DX_CONFIG['end_date']}' THEN 'after end_date' ELSE 'up to end_date' END AS PERIOD,
       COUNT(*) AS ROWS_
FROM O FULL OUTER JOIN N ON {_on}
GROUP BY 1, 2 ORDER BY 1, 2""").toPandas()
print(f"Part A - all US, {CMP_START} to {CMP_END}, rows")
display(_3a)

_b2 = df_dx_basket.select(*_KEYS)
_b3 = df_dx_config.select(*_KEYS)
print("Part B - validation HCPs, rows")
print(f"  Step 2: {_b2.count():,} | Step 3: {_b3.count():,} | only Step 2: {_b2.subtract(_b3).count():,} | only Step 3: {_b3.subtract(_b2).count():,}")
display(_b2.subtract(_b3).groupBy(F.date_format("SVC_DT", "yyyy-MM").alias("MONTH")).count().orderBy("MONTH"))

_3c = Get_Data_Snowflakes(f"""
SELECT 'Step 2 (3.0 dates, demographics join)' AS QUERY, COUNT(*) AS ROWS_PULLED FROM ({dx_query_basket})
UNION ALL
SELECT 'Step 3 (DX_CONFIG dates, no demographics join)', COUNT(*) FROM ({dx_query_config})""").toPandas()
print("Part C - rows returned by each query (all US, before HCP universe and validation filters)")
display(_3c)

### 3e. Findings: Step 3 vs Step 2 (run 2026-10-29)

| Check | Step 2 | Step 3 | Result |
| --- | --- | --- | --- |
| All US, 2025-01 to 2026-09, rows (provider, patient, day, code, stage) | 368.28M | 368.28M | **Identical** (0 rows only in one query) |
| Validation HCPs, rows | 70,892 | 70,892 | **Identical** |
| Rows returned by Snowflake (all US) | 1,086.9M (from 2019-01, date cut in Spark) | 771.6M (from 2023-01-01) | **−29%** |

* `date_end` is 2026-09-30, the same as the end of `current_month` used by 3.0, so the end date changes nothing today. If `date_end` moves inside a month, Step 3 now cuts at that day.
* The demographics join is gone without any change in the output (it was never used).
* Table names, column names and the provider rule are now in `DX_CONFIG`; 3a stops with a clear message if a table or column is missing.

## Step 4: Output columns and dropped-rows report (Q10, Q11)

* The output keeps `DIAGNOSIS_CODE` and adds `CKD_SUBSTAGE` (3a / 3b / ESRD) and `DM_TYPE` from the basket, for QC and for the planned flags (Phases B-C). Rows and stages are the same as Step 3.
* A dropped-rows report counts every source claim row with a staging code under the first rule it fails, so losses are visible on every run.

| Cell | What it does |
| --- | --- |
| 4a | **The Dx query** (Step 3 + basket output columns) and the dropped-rows report SQL |
| 4b | Runs it |
| 4c | Compares with Step 3 and shows the dropped-rows report |
| 4d | Findings |

In [0]:
# 4a. Step 4 query = Step 3 query + basket columns in the output; plus the dropped-rows report.
DX_OUTPUT_BASKET_COLS = ["CKD_SUBSTAGE", "DM_TYPE"]   # basket columns carried to the output

def build_dx_query_step4(basket, cfg, extra_cols=DX_OUTPUT_BASKET_COLS):
    s, p = cfg["source"], cfg["provider"]
    sc, pc = s["columns"], p["columns"]
    prov = _provider_sql(cfg)
    extra = "".join(f",\n       NULLIF(BSK.{c}, '') AS {c}" for c in extra_cols)
    return f"""
SELECT DISTINCT C.{pc['npi']} AS NPI_NUMBER,
       {prov} AS PROVIDER_ID,
       A.{sc['patient_id']} AS PATIENT_ID,
       A.{sc['service_date']} AS SVC_DT,
       A.{sc['diagnosis_code']} AS DIAGNOSIS_CODE,
       BSK.CKD_STAGE AS DX_STAGE{extra}
FROM {s['table']} A
JOIN ({_basket_values(basket, extra_cols)}
) BSK ON A.{sc['diagnosis_code']} = BSK.DIAGNOSIS_CODE
LEFT JOIN {p['table']} C ON {prov} = C.{pc['provider_id']}
WHERE A.{sc['service_date']} BETWEEN '{cfg['start_date']}' AND '{cfg['end_date']}'
  AND {prov} IS NOT NULL
"""

def build_dx_drop_report(basket, cfg):
    # every source claim row with a staging code, under the first rule it fails (rows before DISTINCT)
    s, sc = cfg["source"], cfg["source"]["columns"]
    prov = _provider_sql(cfg)
    d = f"A.{sc['service_date']}"
    return f"""
SELECT REASON, CASE WHEN REASON = 'kept' THEN PROVIDER_ID END AS PROVIDER_ID, COUNT(*) AS ROWS_
FROM (
  SELECT CASE WHEN {d} < '{cfg['start_date']}' THEN '1. before start_date'
              WHEN {d} > '{cfg['end_date']}' THEN '2. after end_date'
              WHEN {prov} IS NULL THEN '3. no provider (referring and rendering empty)'
              ELSE 'kept' END AS REASON,
         {prov} AS PROVIDER_ID
  FROM {s['table']} A
  JOIN ({_basket_values(basket)}
  ) BSK ON A.{sc['diagnosis_code']} = BSK.DIAGNOSIS_CODE
)
GROUP BY 1, 2
"""

dx_query_step4 = build_dx_query_step4(dx_basket, DX_CONFIG)
dx_drop_report_sql = build_dx_drop_report(dx_basket, DX_CONFIG)
print(dx_query_step4)

In [0]:
# 4b. Run the Step 4 query (same post-processing as Step 3)
df_dx_step4_all = Get_Data_Snowflakes(dx_query_step4)
df_dx_step4 = df_dx_step4_all.join(_hcp_universe, how="inner", on="PROVIDER_ID")
df_dx_step4, _ = apply_hcp_filter(df_dx_step4)
print(f"Validation HCPs: {df_dx_step4.count():,} rows, {df_dx_step4.select('PATIENT_ID').distinct().count():,} patients")
display(df_dx_step4.groupBy("DX_STAGE", "CKD_SUBSTAGE", "DM_TYPE").agg(F.count("*").alias("rows"),
        F.countDistinct("PATIENT_ID").alias("patients")).orderBy("DX_STAGE", "CKD_SUBSTAGE", "DM_TYPE"))

In [0]:
# 4c. Part A: Step 4 has the same rows as Step 3 (all US row count + validation HCPs row by row)
#     Part B: dropped-rows report (all US, claim rows with a staging code, first rule that fails)
_4a = Get_Data_Snowflakes(f"""
SELECT (SELECT COUNT(*) FROM ({dx_query_config})) AS STEP3_ROWS,
       (SELECT COUNT(*) FROM (SELECT DISTINCT {', '.join(_KEYS)} FROM ({dx_query_step4}))) AS STEP4_ROWS_SAME_KEYS""").toPandas()
print("Part A - all US rows (Step 4 counted on the Step 3 columns)")
display(_4a)
_c3, _c4 = df_dx_config.select(*_KEYS), df_dx_step4.select(*_KEYS)
print(f"Validation HCPs: only Step 3: {_c3.subtract(_c4).count():,} | only Step 4: {_c4.subtract(_c3).count():,}")

# Part B: dropped-rows report. 'kept' rows are split by the HCP universe (applied in Spark).
_dr = Get_Data_Snowflakes(dx_drop_report_sql)
_kept = (_dr.filter(F.col("REASON") == "kept")
            .join(_hcp_universe.withColumn("_in", F.lit(1)), "PROVIDER_ID", "left")
            .withColumn("REASON", F.when(F.col("_in") == 1, F.lit("kept (provider in HCP universe)"))
                                    .otherwise(F.lit("4. provider not in HCP universe"))))
dx_drop_report = (_dr.filter(F.col("REASON") != "kept").select("REASON", "ROWS_")
                     .unionByName(_kept.select("REASON", "ROWS_"))
                     .groupBy("REASON").agg(F.sum("ROWS_").alias("CLAIM_ROWS"))
                     .withColumn("PCT", F.round(100 * F.col("CLAIM_ROWS") / F.sum("CLAIM_ROWS").over(Window.partitionBy()), 1))
                     .orderBy("REASON"))
print("Part B - dropped-rows report (all US, claim rows with a staging code, before DISTINCT)")
display(dx_drop_report)

### 4d. Findings: Step 4 (run 2026-10-29)

* **Same rows as Step 3:** 771,630,976 rows all US on the Step 3 columns; validation HCPs 70,892 rows, 0 differences.
* **New output columns (validation HCPs):** stage 3 split into 3a (400 patients), 3b (378) and unspecified N18.30 / N18.3 (258); stage 5 = ESRD N18.6 (292 patients) and other stage-5 codes (226). `DM_TYPE` on stage-0 diabetic kidney codes: T2D 588, T1D 21, OTHER 14 patients.

**Dropped-rows report (all US, claim rows with a staging code, before DISTINCT)**

| Rule | Claim rows | % |
| --- | --- | --- |
| 1. Before `start_date` (2023-01-01) | 399.1M | 27.2 |
| 2. After `end_date` (2026-09-30) | 0.55M | 0.0 |
| 3. No provider (referring and rendering both empty) | 178.3M | 12.1 |
| 4. Provider not in the HCP universe | 365.6M | 24.9 |
| Kept | 524.4M | 35.7 |

* Rules 1-2 are by design. Rule 3 is the 3.0 rule: 12% of CKD claim rows have no doctor and cannot be linked to an HCP (decision N3 keeps it). Rule 4 is the 3.0 HCP universe filter.
* The report runs on every execution of 4c, so a sudden change in any line shows up.

## Step 5: Processing fixes (C1-C3, C5; decisions N1, N2)

New monthly processing `dx_monthly_31`, applied to the Step 4 query output. Compared with `dx_monthly_30`:

| 3.0 | 3.1 | Why |
| --- | --- | --- |
| Global grid: every patient x every month between the first and last service date of all patients | **Per patient**: from the cohort month of the first CKD Dx to the cohort month of `date_end` (N2) | No rows before the first diagnosis; far smaller at all-US scale; every patient reaches the latest month |
| `DX_STAGE_UPDATED = 0` for both "CKD, stage unknown" and "before any CKD Dx" | `DX_STAGE_UPDATED = null` when no stage is known; new `DX_CKD_FLAG = 1` on every month from the first CKD Dx (N1) | In PJ06 `COALESCE(DX_STAGE_UPDATED, GFR_STAGE)` a 0 hid the lab stage; null lets the lab stage through, with no change in PJ06 |
| `DX_STAGE` = array of the stages in the month | `DX_STAGE_MONTH` = highest stage coded in the month (0 = CKD code without stage; null = no CKD claim that month) | Easier to use |
| - | `DX_SUBSTAGE_UPDATED` (3a / 3b / ESRD), carried with the stage | Plan Phase A |

Kept from 3.0: highest stage within a month; the latest staged month is carried forward (so a later lower stage replaces an earlier higher one); cohort month assignment with `OFFSET_DAY`; cohort label.

| Cell | What it does |
| --- | --- |
| 5a | `dx_monthly_31` and the result for validation HCPs (`df_dx_combine`) |
| 5b | Compares 3.0 vs 3.1 processing on the same input |
| 5c | Findings |

In [0]:
# 5a. 3.1 monthly processing. Input: PATIENT_ID, SVC_DT, DX_STAGE (+ CKD_SUBSTAGE) from the Step 4 query.
# Output, one row per patient x month from the first CKD Dx month to the date_end cohort month:
#   PATIENT_ID, DATE, DX_STAGE_MONTH, DX_STAGE_UPDATED, DX_SUBSTAGE_UPDATED, DX_CKD_FLAG, cohort_label

def dx_monthly_31(df, end_date=DX_CONFIG["end_date"]):
    d = (df.withColumn("PATIENT_ID", F.col("PATIENT_ID").cast("string"))
           .withColumn("DATE", F.to_date(F.date_format(assign_cohort_month(F.col("SVC_DT")), "yyyy-MM-01")))
           .withColumn("DX_STAGE", F.col("DX_STAGE").cast("int")))
    # highest stage in the month; the substage goes with that stage (e.g. ESRD with 5, 3b over 3a)
    monthly = (d.groupBy("PATIENT_ID", "DATE")
                .agg(F.max(F.struct("DX_STAGE", F.coalesce("CKD_SUBSTAGE", F.lit("")).alias("SUB"))).alias("_m"))
                .select("PATIENT_ID", "DATE", F.col("_m.DX_STAGE").alias("DX_STAGE_MONTH"),
                        F.when(F.col("_m.SUB") != "", F.col("_m.SUB")).alias("_SUB")))

    # per-patient month range: first CKD Dx month -> cohort month of end_date (N2)
    last_month = F.to_date(F.date_format(assign_cohort_month(F.to_date(F.lit(end_date))), "yyyy-MM-01"))
    grid = (monthly.groupBy("PATIENT_ID").agg(F.min("DATE").alias("_first"))
                   .select("PATIENT_ID", F.explode(F.sequence("_first", last_month, F.expr("INTERVAL 1 MONTH"))).alias("DATE")))
    out = grid.join(monthly, ["PATIENT_ID", "DATE"], "left")

    # carry the latest known stage forward; 0 = stage unknown, not carried (N1: null instead of 0)
    w = Window.partitionBy("PATIENT_ID").orderBy("DATE").rowsBetween(Window.unboundedPreceding, 0)
    _known = F.when(F.col("DX_STAGE_MONTH") > 0, F.struct(F.col("DX_STAGE_MONTH").alias("S"), F.col("_SUB").alias("SUB")))
    out = (out.withColumn("_f", F.last(_known, ignorenulls=True).over(w))
              .withColumn("DX_STAGE_UPDATED", F.col("_f.S"))
              .withColumn("DX_SUBSTAGE_UPDATED", F.col("_f.SUB"))
              .withColumn("DX_CKD_FLAG", F.lit(1))
              .drop("_f", "_SUB"))
    out = out.join(df_cohort_ref.select(F.to_date(F.date_format("cohort_ref_month", "yyyy-MM-01")).alias("DATE"), "cohort_label"),
                   "DATE", "left")
    return out.select("PATIENT_ID", "DATE", "DX_STAGE_MONTH", "DX_STAGE_UPDATED", "DX_SUBSTAGE_UPDATED", "DX_CKD_FLAG", "cohort_label")

df_dx_combine = dx_monthly_31(df_dx_step4)

# Checks: one row per patient-month; no gaps; every patient ends at the last month; stage only 1-5 or null
_n, _nk = df_dx_combine.count(), df_dx_combine.select("PATIENT_ID", "DATE").distinct().count()
assert _n == _nk, f"Duplicate patient-months: {_n - _nk}"
_span = df_dx_combine.groupBy("PATIENT_ID").agg(F.count("*").alias("n"), F.min("DATE").alias("a"), F.max("DATE").alias("b"))
assert _span.filter(F.col("n") != F.months_between("b", "a") + 1).count() == 0, "Gaps in a patient's months"
assert _span.select("b").distinct().count() == 1, "Patients end in different months"
assert df_dx_combine.filter(~F.col("DX_STAGE_UPDATED").between(1, 5)).count() == 0, "DX_STAGE_UPDATED outside 1-5"
print(f"df_dx_combine: {_n:,} patient-months, {df_dx_combine.select('PATIENT_ID').distinct().count():,} patients, "
      f"last month {_span.select('b').first()[0]}")
display(df_dx_combine.groupBy("DX_STAGE_UPDATED", "DX_SUBSTAGE_UPDATED").agg(F.count("*").alias("patient_months"),
        F.countDistinct("PATIENT_ID").alias("patients")).orderBy("DX_STAGE_UPDATED", "DX_SUBSTAGE_UPDATED"))
display(df_dx_combine.orderBy("PATIENT_ID", "DATE").limit(30))

In [0]:
# 5b. Same input (Step 4 query, validation HCPs) through the 3.0 processing and the 3.1 processing.
_p30 = dx_monthly_30(df_dx_step4).select("PATIENT_ID", "DATE", F.col("DX_STAGE_UPDATED").cast("int").alias("S30"),
                                         F.col("cohort_label").alias("L30"))
_p31 = df_dx_combine.select("PATIENT_ID", "DATE", F.col("DX_STAGE_UPDATED").alias("S31"), "DX_STAGE_MONTH",
                            F.col("cohort_label").alias("L31"))
_pj = _p30.join(_p31, ["PATIENT_ID", "DATE"], "full")
_first = df_dx_combine.groupBy("PATIENT_ID").agg(F.min("DATE").alias("_first"))
_pj = _pj.join(_first, "PATIENT_ID", "left")
_cat = (F.when(F.col("S31").isNull() & F.col("L31").isNull() & F.col("DX_STAGE_MONTH").isNull() & (F.col("DATE") < F.col("_first")),
               F.lit("only 3.0: before first CKD Dx (3.0 stage 0)"))
         .when(F.col("S30").isNull() & F.col("L30").isNull(), F.lit("only 3.1: after the last service date of all patients"))
         .when(F.col("S30") == F.col("S31"), F.lit("same stage"))
         .when((F.col("S30") == 0) & F.col("S31").isNull(), F.lit("3.0 = 0, 3.1 = null (CKD, stage unknown)"))
         .otherwise(F.lit("different stage")))
print("Validation HCPs, patient-months, same input through both processings")
print(f"  3.0: {_p30.count():,} patient-months | 3.1: {_p31.count():,} patient-months")
display(_pj.groupBy(_cat.alias("RESULT")).agg(F.count("*").alias("patient_months"), F.countDistinct("PATIENT_ID").alias("patients"))
           .orderBy("RESULT"))
_lbl_diff = _pj.filter(F.col("S30").isNotNull() & F.col("L31").isNotNull() & ~F.col("L30").eqNullSafe(F.col("L31"))).count()
print(f"  cohort label differs on common months: {_lbl_diff:,}")

# Effect on PJ06: COALESCE(DX_STAGE_UPDATED, GFR_STAGE). Months where the 3.0 Dx value 0 hid the lab stage.
_hidden = _pj.filter((F.col("S30") == 0)).select("PATIENT_ID", "DATE")
print(f"  months where 3.0 gives 0 (would hide a lab stage in PJ06): {_hidden.count():,}, "
      f"of which inside the 3.1 range (lab stage now used): {_hidden.join(_p31, ['PATIENT_ID', 'DATE']).count():,}")

### 5c. Findings: 3.1 processing (run 2026-10-29, validation HCPs, Step 4 input)

| Result | Patient-months | Patients |
| --- | --- | --- |
| Same stage (1-5) | 33,102 | 1,030 |
| 3.0 = 0, 3.1 = null (CKD coded, stage unknown) | 5,873 | 334 |
| Only 3.0: months before the patient's first CKD Dx | 14,523 | 1,059 |
| Only 3.1 | 0 | |
| Different stage | 0 | |

* **Totals reconcile:** 3.0 = 33,102 + 5,873 + 14,523 = 53,498; 3.1 = 33,102 + 5,873 = 38,975 (−27%). Cohort labels are identical on all common months.
* **No stage changes** where a stage is known: the only differences are the rows removed before the first diagnosis (N2) and 0 becoming null (N1).
* **Effect on PJ06:** in the 5,873 months where the patient has a CKD code without a stage, `COALESCE(DX_STAGE_UPDATED, GFR_STAGE)` will now use the lab stage instead of stopping at 0. The 14,523 months before the first diagnosis disappear from the Dx journey; PJ06 must keep lab-only months through its join (to check in PJ06).
* All patients end in 2026-09 (cohort month of `date_end`); no gaps; one row per patient-month (checked in 5a).
* New columns: `DX_STAGE_MONTH`, `DX_SUBSTAGE_UPDATED` (3a 379, 3b 364, ESRD 292 patients reach these), `DX_CKD_FLAG`. `DX_STAGE` (array) is no longer produced.

**Kept from 3.0 for review:** the latest staged month wins (a later stage 3 replaces an earlier stage 4).

## 2. Step 6: Write the Dx journey (after review only)

`df_dx_combine` (Step 4 query + 3.1 processing, validation HCPs) is ready. The write is switched off (`WRITE_OUTPUT = False`) until the steps above are reviewed. Before switching it on: confirm PJ06 still keeps lab-only months (see 5c), and note that the columns change (`DX_STAGE` array removed; `DX_STAGE_MONTH`, `DX_SUBSTAGE_UPDATED`, `DX_CKD_FLAG` added).

In [0]:
# Step 6: write the 3.1 Dx journey. Switched off until review.
WRITE_OUTPUT = False
DX_OUTPUT_PATH = DX_VALIDATION_PATH   # kerendia_nba_3_1_dx_journey_validation (same table as the old notebook)

print(f"Output: {df_dx_combine.count():,} patient-months, {df_dx_combine.select('PATIENT_ID').distinct().count():,} patients")
if WRITE_OUTPUT:
    (df_dx_combine.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(DX_OUTPUT_PATH))
    print(f"Written to {DX_OUTPUT_PATH}")
else:
    print("WRITE_OUTPUT = False: nothing written (set to True after review)")